# Musical Congruence Pipeline
**Sound and Sense: Multimodal Emotional Congruence in Music and Its Effect on Streaming Popularity**

This notebook runs the full pipeline end-to-end by sourcing each step script rather than duplicating their code.

| Step | Script | Description |
|------|--------|-------------|
| 1 | `01_scrape_lyrics.py` | Genius API → lyrics JSON |
| 2 | `02_scrape_audio.py` | yt-dlp → MP3s |
| 3 | `03_build_schema.py` | Auto-generate schema `.py` files |
| 4 | `04_embed_mulan.py` | MuQ-MuLan embeddings |
| 5 | `05_embed_clap.py` | LAION-CLAP-Music embeddings |
| 6 | `06_embed_mert_sbert.py` | MERT + Sentence-BERT (late-fusion baseline) |
| 7 | `07_segment_analysis.py` | Verse/chorus/bridge segment-level LMC |
| 8 | `08_get_popularity.py` | Spotify popularity + audio features |
| 9 | `09_combine_results.py` | Merge all outputs → `master_results.csv` |

---
## 0. Environment Setup
Sets the working directory, confirms API keys are present, and defines a helper that loads numbered scripts via `importlib`.

In [1]:
import os
import getpass

os.environ["GENIUS_API_TOKEN"]      = getpass.getpass("Genius API Token: ")
os.environ["SPOTIFY_CLIENT_ID"]     = getpass.getpass("Spotify Client ID: ")
os.environ["SPOTIFY_CLIENT_SECRET"] = getpass.getpass("Spotify Client Secret: ")

# Force config to pick up the new values
import config
config.GENIUS_API_TOKEN      = os.environ["GENIUS_API_TOKEN"]
config.SPOTIFY_CLIENT_ID     = os.environ["SPOTIFY_CLIENT_ID"]
config.SPOTIFY_CLIENT_SECRET = os.environ["SPOTIFY_CLIENT_SECRET"]

print("Keys set ✓")

Genius API Token:  ········
Spotify Client ID:  ········
Spotify Client Secret:  ········


Keys set ✓


In [2]:
import sys
import importlib.util

# ── Working directory ─────────────────────────────────────────────────────────
# Point this at the folder containing config.py and the 0N_*.py scripts.
PROJECT_DIR = os.path.dirname(os.path.abspath("__file__"))  # same dir as this notebook
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"Working directory: {os.getcwd()}")

# ── API key check ─────────────────────────────────────────────────────────────
REQUIRED_KEYS = ["GENIUS_API_TOKEN", "SPOTIFY_CLIENT_ID", "SPOTIFY_CLIENT_SECRET"]
for key in REQUIRED_KEYS:
    val = os.environ.get(key, "")
    status = "✓ set" if val and "YOUR_" not in val else "✗ MISSING"
    print(f"  {key:35s} {status}")

# ── Script loader ─────────────────────────────────────────────────────────────
# Python cannot import files whose names start with a digit, so we use importlib.
def load_script(alias, filename):
    """Load a pipeline script by filename and register it under `alias`."""
    path = os.path.join(PROJECT_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Script not found: {path}")
    spec = importlib.util.spec_from_file_location(alias, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[alias] = mod
    spec.loader.exec_module(mod)
    return mod

# ── Device report ─────────────────────────────────────────────────────────────
import torch
if   torch.cuda.is_available():              device = "cuda"
elif torch.backends.mps.is_available():      device = "mps"
else:                                        device = "cpu"
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
print(f"\nDevice: {device}")

# ── Catalog summary ───────────────────────────────────────────────────────────
from config import CATALOG, total_songs
print(f"Catalog: {len(CATALOG)} artists, {total_songs()} songs")

Working directory: /Users/budge.13/Desktop/Music Analysis
  GENIUS_API_TOKEN                    ✓ set
  SPOTIFY_CLIENT_ID                   ✓ set
  SPOTIFY_CLIENT_SECRET               ✓ set

Device: mps
Catalog: 34 artists, 441 songs


---
## Step 1 — Scrape Lyrics
Pulls lyrics from the Genius API for every song in the catalog.  
**Outputs:** `lyrics/{SONG_ID}.json` for each song, `lyrics/_manifest.json`  
**Resume-safe:** songs already in `lyrics/` are skipped automatically.

> Set `artist_filter` to a list like `["KL", "GD"]` to restrict to specific artists.

In [3]:
scrape_lyrics = load_script("scrape_lyrics", "01_scrape_lyrics.py")

manifest_lyrics = scrape_lyrics.scrape_all(
    dry_run       = False,   # set True to preview without fetching
    artist_filter = None,    # e.g. ["KL", "GD", "TS"] — None = all artists
)

print(f"\nFetched : {manifest_lyrics['n_fetched']}")
print(f"Skipped : {manifest_lyrics['n_skipped']}")
print(f"Failed  : {manifest_lyrics['n_failed']}")
if manifest_lyrics['failed']:
    print(f"Failed IDs: {manifest_lyrics['failed']}")

23:02:55 | INFO | scrape_lyrics | 
────────────────────────────────────────────────────────────
23:02:55 | INFO | scrape_lyrics | Artist: Run the Jewels  (RTJ)
23:02:55 | INFO | scrape_lyrics |   [RTJ_01] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_02] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_03] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_04] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_05] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_06] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_07] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_08] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_09] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_10] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_11] SKIP (already fetched)
23:02:55 | INFO | scrape_lyrics |   [RTJ_12] SKIP (already fetched)
23:02:55 | INFO | scrape


Fetched : 263
Skipped : 174
Failed  : 4
Failed IDs: ['TW_13', 'DP_04', 'CB_06', 'BN_12']


---
## Step 2 — Scrape Audio
Downloads each song as a 192 kbps MP3 from YouTube via yt-dlp.  
**Outputs:** `audio/{ARTIST_FOLDER}/{SONG_ID}.mp3`  
**Resume-safe:** existing files are skipped. Failures create a `.FAILED` stub.  

> **Time estimate:** ~90–120 seconds per artist (10–15 songs × ~8s per download + rate-limit delay).  
> For a quick test, set `artist_filter=["KL", "GD", "TS"]` (~3 artists, ~45 songs).

In [4]:
scrape_audio = load_script("scrape_audio", "02_scrape_audio.py")

manifest_audio = scrape_audio.download_all(
    dry_run       = False,
    artist_filter = None,    # e.g. ["KL", "GD", "TS"]
    max_retries   = 2,
)

print(f"\nDownloaded : {manifest_audio['n_downloaded']}")
print(f"Skipped    : {manifest_audio['n_skipped']}")
print(f"Failed     : {manifest_audio['n_failed']}")

# Coverage report
report = scrape_audio.get_coverage_report()
print(f"\nCoverage   : {len(report['have_audio'])} / {total_songs()} songs have audio")

23:15:20 | INFO | scrape_audio | 
────────────────────────────────────────────────────────────
23:15:20 | INFO | scrape_audio | Artist: Run the Jewels  (RTJ)
23:15:20 | INFO | scrape_audio |   [RTJ_01] SKIP — /Users/budge.13/Desktop/Music Analysis/audio/RTJ/RTJ_01.mp3
23:15:20 | INFO | scrape_audio |   [RTJ_02] SKIP — /Users/budge.13/Desktop/Music Analysis/audio/RTJ/RTJ_02.mp3
23:15:20 | INFO | scrape_audio |   [RTJ_03] SKIP — /Users/budge.13/Desktop/Music Analysis/audio/RTJ/RTJ_03.mp3
23:15:20 | INFO | scrape_audio |   [RTJ_04] SKIP — /Users/budge.13/Desktop/Music Analysis/audio/RTJ/RTJ_04.mp3
23:15:20 | INFO | scrape_audio |   [RTJ_05] SKIP — /Users/budge.13/Desktop/Music Analysis/audio/RTJ/RTJ_05.mp3
23:15:20 | INFO | scrape_audio |   [RTJ_06] SKIP — /Users/budge.13/Desktop/Music Analysis/audio/RTJ/RTJ_06.mp3
23:15:20 | INFO | scrape_audio |   [RTJ_07] SKIP — /Users/budge.13/Desktop/Music Analysis/audio/RTJ/RTJ_07.mp3
23:15:20 | INFO | scrape_audio |   [RTJ_08] SKIP — /Users/budge.1

23:15:28 | INFO | scrape_audio |     ✓ Downloaded: 'God's Plan' (199s)
23:15:32 | INFO | scrape_audio |   [DR_02] Searching: Drake Hotline Bling official audio


23:15:37 | INFO | scrape_audio |     ✓ Downloaded: 'Drake - Hotline Bling (Official Audio)' (267s)
23:15:39 | INFO | scrape_audio |   [DR_03] Searching: Drake One Dance official audio


23:15:48 | INFO | scrape_audio |     ✓ Downloaded: 'One Dance (feat. WizKid & Kyla) - Drake (Official Audio)' (174s)
23:15:51 | INFO | scrape_audio |   [DR_04] Searching: Drake Passionfruit official audio


23:15:58 | INFO | scrape_audio |     ✓ Downloaded: 'Passionfruit' (299s)
23:16:00 | INFO | scrape_audio |   [DR_05] Searching: Drake Started From the Bottom audio


...

00:09:23 | INFO | scrape_audio |     ✓ Downloaded: 'Gorillaz - Andromeda (DRAM Special) [Audio]' (238s)
00:09:26 | INFO | scrape_audio |   [GZ_10] Searching: Gorillaz Tranz official audio


00:09:32 | INFO | scrape_audio |     ✓ Downloaded: 'Gorillaz - Tranz (Official Visualiser)' (162s)
00:09:35 | INFO | scrape_audio |   [GZ_11] Searching: Gorillaz Hollywood Snoop Dogg Jamie Principle audio


00:09:44 | INFO | scrape_audio |     ✓ Downloaded: 'Gorillaz - Hollywood feat. Snoop Dogg & Jamie Principle (Official Visualiser)' (296s)
00:09:47 | INFO | scrape_audio |   [GZ_12] Searching: Gorillaz New Gold Tame Impala Bootie Brown audio


00:09:52 | INFO | scrape_audio |     ✓ Downloaded: 'Gorillaz - New Gold ft. Tame Impala & Bootie Brown (Official Visualiser)' (216s)
00:09:56 | INFO | scrape_audio | 
════════════════════════════════════════════════════════════
00:09:56 | INFO | scrape_audio | Done. Downloaded: 264  |  Skipped: 176  |  Failed: 1
00:09:56 | WARNING | scrape_audio | Failed songs (check .FAILED stubs): ['DS_10']
00:09:56 | INFO | scrape_audio | Manifest → /Users/budge.13/Desktop/Music Analysis/audio/_manifest.json



Downloaded : 264
Skipped    : 176
Failed     : 1

Coverage   : 440 / 441 songs have audio


---
## Step 3 — Build Schemas
Reads each `lyrics/{SONG_ID}.json` and generates per-artist schema `.py` files  
(`schemas/KL_schema.py`, etc.) in the same format as the original `RTJ.py` / `GD.py`.  
Also writes `schemas/full_catalog.py` which merges all schemas.  
**Outputs:** `schemas/{ARTIST_CODE}_schema.py`, `schemas/full_catalog.py`

In [5]:
build_schema = load_script("build_schema", "03_build_schema.py")

schema_summary = build_schema.build_all_schemas(artist_filter=None)

total_included = sum(len(v["included"]) for v in schema_summary.values())
total_missing  = sum(len(v["missing"])  for v in schema_summary.values())
print(f"\nSchema build: {total_included} songs included, {total_missing} missing lyrics")

00:09:56 | INFO | build_schema | Building schema for Run the Jewels (RTJ)…
00:09:56 | INFO | build_schema |   → /Users/budge.13/Desktop/Music Analysis/schemas/RTJ_schema.py  (15 songs, 0 missing)
00:09:56 | INFO | build_schema | Building schema for Kendrick Lamar (KL)…
00:09:56 | INFO | build_schema |   → /Users/budge.13/Desktop/Music Analysis/schemas/KL_schema.py  (15 songs, 0 missing)
00:09:56 | INFO | build_schema | Building schema for J. Cole (JC)…
00:09:56 | INFO | build_schema |   → /Users/budge.13/Desktop/Music Analysis/schemas/JC_schema.py  (15 songs, 0 missing)
00:09:56 | INFO | build_schema | Building schema for Drake (DR)…
00:09:56 | INFO | build_schema |   → /Users/budge.13/Desktop/Music Analysis/schemas/DR_schema.py  (12 songs, 0 missing)
00:09:56 | INFO | build_schema | Building schema for Eminem (EM)…
00:09:56 | INFO | build_schema |   → /Users/budge.13/Desktop/Music Analysis/schemas/EM_schema.py  (12 songs, 0 missing)
00:09:56 | INFO | build_schema | Building schema for


Schema build: 437 songs included, 4 missing lyrics


---
## Step 4 — MuQ-MuLan Embeddings
Computes audio and text embeddings using the jointly-trained MuQ-MuLan model.  
**Model:** `OpenMuQ/MuQ-MuLan-large` — downloads ~1.5 GB on first run.  
**Outputs:** `results/embeddings/mulan/{audio,text}_embeddings.npz`, `similarities.json`  
**Resume-safe:** `checkpoint.json` tracks completed songs.

In [6]:
embed_mulan = load_script("embed_mulan", "04_embed_mulan.py")

mulan_results = embed_mulan.embed_all(
    device = device,
    force  = False,   # set True to recompute everything from scratch
)

if mulan_results:
    sims = mulan_results["sims"]
    vals = list(sims.values())
    import numpy as np
    print(f"\nMuLan LMC  — n={len(vals)}  mean={np.mean(vals):.4f}  "
          f"sd={np.std(vals):.4f}  min={np.min(vals):.4f}  max={np.max(vals):.4f}")

00:09:56 | INFO | embed_mulan | Loading model for 263 songs…
00:10:00 | INFO | numexpr.utils | NumExpr defaulting to 10 threads.
00:10:01 | INFO | embed_mulan | Loading MuQ-MuLan on mps…
00:10:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
00:10:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/OpenMuQ/MuQ-MuLan-large/2e01c796b71dca71b45251384c04cd7b237c9020/config.json "HTTP/1.1 200 OK"
00:10:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-large-msd-iter/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
00:10:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/OpenMuQ/MuQ-large-msd-iter/0562a57814f6f8bbd9fdea0a25921a2fce1a841a/config.json "HTTP/1.1 200 OK"
/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is dep

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
00:10:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
00:10:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
00:10:08 | INFO | embed_mulan | [1/263] DR_01
00:10:13 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
00:10:13 | INFO | httpx | HTTP Request: HEAD ht


MuLan LMC  — n=436  mean=0.3318  sd=0.0930  min=0.0114  max=0.6435


---
## Step 5 — LAION-CLAP-Music Embeddings
Computes embeddings using the contrastive audio-language model trained specifically on music.  
**Model:** `laion/larger_clap_music` — downloads ~600 MB on first run.  
Audio is processed in 10-second overlapping chunks and averaged.  
**Outputs:** `results/embeddings/clap/{audio,text}_embeddings.npz`, `similarities.json`  

> If this step is very slow on MPS, change `device` to `"cpu"` in the cell below.

In [7]:
embed_clap = load_script("embed_clap", "05_embed_clap.py")

clap_results = embed_clap.embed_all(
    device = device,   # swap to "cpu" if MPS is slow for this model
    force  = False,
)

if clap_results:
    sims = clap_results["sims"]
    vals = list(sims.values())
    print(f"\nCLAP LMC   — n={len(vals)}  mean={np.mean(vals):.4f}  "
          f"sd={np.std(vals):.4f}  min={np.min(vals):.4f}  max={np.max(vals):.4f}")

01:30:56 | INFO | embed_clap | Loading CLAP model for 263 songs…
01:30:57 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
01:30:57 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
01:30:57 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
01:30:57 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
01:30:57 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
01:30:57 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/rob

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
01:31:00 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
01:31:00 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/roberta-base/tre

Load the specified checkpoint /Users/budge.13/.cache/huggingface/hub/models--lukewys--laion_clap/snapshots/b3708341862f581175dba5c356a4ebf74a9b6651/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...


01:31:02 | INFO | embed_clap | [1/263] DR_01


logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm2.bias 	 Loaded
audio_branch.layers.0.blocks.0

01:31:03 | INFO | embed_clap | [2/263] DR_02
01:31:03 | INFO | embed_clap | [3/263] DR_03
01:31:03 | INFO | embed_clap | [4/263] DR_04
01:31:04 | INFO | embed_clap | [5/263] DR_05
01:31:04 | INFO | utils | Saved embeddings → /Users/budge.13/Desktop/Music Analysis/results/embeddings/clap/audio_embeddings.npz
01:31:04 | INFO | utils | Saved embeddings → /Users/budge.13/Desktop/Music Analysis/results/embeddings/clap/text_embeddings.npz
01:31:04 | INFO | embed_clap |   Checkpoint (5 done)
01:31:04 | INFO | embed_clap | [6/263] DR_06
01:31:04 | INFO | embed_clap | [7/263] DR_07
01:31:05 | INFO | embed_clap | [8/263] DR_08
01:31:05 | INFO | embed_clap | [9/263] DR_09
01:31:06 | INFO | embed_clap | [10/263] DR_10
01:31:07 | INFO | utils | Saved embeddings → /Users/budge.13/Desktop/Music Analysis/results/embeddings/clap/audio_embeddings.npz
01:31:07 | INFO | utils | Saved embeddings → /Users/budge.13/Desktop/Music Analysis/results/embeddings/clap/text_embeddings.npz
01:31:07 | INFO | embed_cla


CLAP LMC   — n=436  mean=0.0744  sd=0.0998  min=-0.2297  max=0.4262


---
## Step 6 — MERT + Sentence-BERT (Late-Fusion Baseline)
Computes audio embeddings (MERT) and text embeddings (Sentence-BERT) **independently**,  
then takes cosine similarity across the two separate spaces.  
This is the critical baseline: if jointly-trained models (Steps 4–5) outperform this,  
it confirms that the shared embedding space — not just acoustic or semantic content alone — drives LMC predictive validity.  
**Models:** `m-a-p/MERT-v1-95M` (~380 MB) + `all-mpnet-base-v2` (~420 MB)  
**Outputs:** `results/embeddings/mert_sbert/{audio,text}_embeddings.npz`, `similarities.json`

In [8]:
embed_mert = load_script("embed_mert_sbert", "06_embed_mert_sbert.py")

mert_results = embed_mert.embed_all(
    device = device,
    force  = False,
)

if mert_results:
    sims = mert_results["sims"]
    vals = list(sims.values())
    print(f"\nMERT+SBERT — n={len(vals)}  mean={np.mean(vals):.4f}  "
          f"sd={np.std(vals):.4f}  min={np.min(vals):.4f}  max={np.max(vals):.4f}")

01:33:01 | INFO | embed_mert_sbert | Loading models for 263 songs…
01:33:01 | INFO | embed_mert_sbert | Loading MERT (m-a-p/MERT-v1-95M) on mps…
01:33:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/m-a-p/MERT-v1-95M/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
01:33:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/m-a-p/MERT-v1-95M/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
01:33:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/m-a-p/MERT-v1-95M/12af15fef9d0ac838c3f475bfbbf26d2060dd4f5/preprocessor_config.json "HTTP/1.1 200 OK"
01:33:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/m-a-p/MERT-v1-95M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
01:33:01 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/m-a-p/MERT-v1-95M/12af15fef9d0ac838c3f475bfbbf26d2060dd4f5/config.json "HTTP/1.1 200 OK"
01:33:01 | INFO | httpx | HTT

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

01:33:03 | INFO | embed_mert_sbert | Loading SBERT (sentence-transformers/all-mpnet-base-v2) on mps…
01:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
01:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
01:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
01:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config_sentence_transformers.json "HTTP/1.1 200 OK"
01:33:03 | INFO | sentence_transformers.base.model | Loading SentenceTransformer model from sentence-transforme

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
01:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
01:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
01:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
01:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not F


MERT+SBERT — n=436  mean=-0.0274  sd=0.0302  min=-0.1190  max=0.0785


---
## Step 7 — Segment-Level Analysis *(optional)*
Parses lyrics into sections (`[Verse]`, `[Chorus]`, `[Bridge]`, etc.) using Genius section headers,  
splits the audio into chunks aligned to sections using WhisperX,  
and computes per-section MuLan LMC.  
**Outputs:** `results/segment_analysis/segment_details.json`, `segment_summary.csv`  

> This step is compute-intensive (~2–4 min per song). Set `enabled=False` in `config.py`  
> or skip this cell if you only need track-level results.

In [9]:
segment_analysis = load_script("segment_analysis", "07_segment_analysis.py")

seg_details, seg_summary = segment_analysis.run_segment_analysis(
    device        = device,
    force         = False,
    artist_filter = None,   # e.g. ["KL", "GD"] to limit scope
)

n_analysed = sum(1 for v in seg_details.values() if v)
print(f"\nSegment analysis complete: {n_analysed} songs with valid section data")
if seg_summary:
    import pandas as pd
    seg_df = pd.DataFrame(seg_summary)
    print(f"Mean sections per song : {seg_df['n_segments'].mean():.1f}")
    print(f"Mean segment LMC (all) : {seg_df['mean_lmc_all'].mean():.4f}")

13:11:34 | INFO | segment_analysis | Resuming — 173 songs already done.
13:11:34 | INFO | segment_analysis | Loading models for 263 songs...
13:11:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
13:11:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/OpenMuQ/MuQ-MuLan-large/2e01c796b71dca71b45251384c04cd7b237c9020/config.json "HTTP/1.1 200 OK"
13:11:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-large-msd-iter/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
13:11:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/OpenMuQ/MuQ-large-msd-iter/0562a57814f6f8bbd9fdea0a25921a2fce1a841a/config.json "HTTP/1.1 200 OK"
/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametriz

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
13:11:41 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
13:11:41 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
13:12:14 | INFO | segment_analysis | Loading WhisperX (whisper=base, align on cpu)...
/opt/anaconda3/lib/python3.13/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-

2026-04-20 13:12:18 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-04-20 13:12:18 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../opt/anaconda3/lib/python3.13/site-packages/whisperx/assets/pytorch_model.bin`
13:12:19 | INFO | lightning.pytorch.utilities.migration.utils | Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../opt/anaconda3/lib/python3.13/site-packages/whisperx/assets/pytorch_model.bin`
13:12:19 | INFO | segment_analysis | [1/263] DR_01


...

2026-04-20 15:58:12 - whisperx.asr - INFO - Detected language: en (0.79) in first 30s of audio


15:58:30 | INFO | segment_analysis |   ✓ 3 sections | mean LMC=0.3028 | method={'whisperx'}
15:58:30 | INFO | segment_analysis | [263/263] GZ_12
python(49533) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


2026-04-20 15:58:45 - whisperx.asr - INFO - Detected language: en (0.33) in first 30s of audio


15:59:18 | INFO | segment_analysis |   ✓ 3 sections | mean LMC=0.1945 | method={'whisperx'}
15:59:18 | INFO | segment_analysis |   Checkpoint saved.
15:59:18 | INFO | segment_analysis | 
Segment summary: 383 songs (372 WhisperX-aligned, 11 equal-split fallback)
15:59:18 | INFO | segment_analysis | → /Users/budge.13/Desktop/Music Analysis/results/segment_analysis/segment_summary.csv



Segment analysis complete: 383 songs with valid section data
Mean sections per song : 4.0
Mean segment LMC (all) : 0.3004


---
## Step 8 — Spotify Popularity
Searches the Spotify API for each song and fetches the popularity score (0–100)  
plus Echo Nest audio features (energy, danceability, valence, etc.) that serve as controls.  
**Outputs:** `results/spotify_data.json`, `results/spotify_summary.csv`  
**Resume-safe:** already-fetched songs are skipped.

In [10]:
get_popularity = load_script("get_popularity", "08_get_popularity.py")

spotify_data = get_popularity.fetch_all_popularity(force=False)

n_found   = sum(1 for v in spotify_data.values() if v.get("found"))
n_missing = len(spotify_data) - n_found
print(f"\nSpotify match rate: {n_found}/{len(spotify_data)} songs found")
if n_missing:
    missing_ids = [k for k, v in spotify_data.items() if not v.get("found")]
    print(f"Not found: {missing_ids}")

import pandas as pd
pop_vals = [v["popularity"] for v in spotify_data.values() if v.get("found")]
if pop_vals:
    import numpy as np
    print(f"Popularity — mean={np.mean(pop_vals):.1f}  "
          f"sd={np.std(pop_vals):.1f}  "
          f"min={np.min(pop_vals)}  max={np.max(pop_vals)}")

15:59:19 | INFO | get_popularity | Loaded existing Spotify data for 177 songs.
15:59:19 | INFO | get_popularity | 
───────────────────────────────────────────────────────  Run the Jewels
15:59:19 | INFO | get_popularity | 
───────────────────────────────────────────────────────  Kendrick Lamar
15:59:19 | INFO | get_popularity | 
───────────────────────────────────────────────────────  J. Cole
15:59:19 | INFO | get_popularity | 
───────────────────────────────────────────────────────  Drake
15:59:19 | INFO | get_popularity |   [DR_01] Searching: God's Plan
15:59:19 | INFO | get_popularity |     ✓ Found: 'God's Plan' | popularity=86
15:59:20 | INFO | get_popularity |   [DR_02] Searching: Hotline Bling
15:59:20 | INFO | get_popularity |     ✓ Found: 'Hotline Bling' | popularity=85
15:59:20 | INFO | get_popularity |   [DR_03] Searching: One Dance
15:59:21 | INFO | get_popularity |     ✓ Found: 'One Dance' | popularity=92
15:59:21 | INFO | get_popularity |   [DR_04] Searching: Passionfruit



Spotify match rate: 363/441 songs found
Not found: ['TC2_08', 'TC2_09', 'TC2_10', 'TC2_11', 'TC2_12', 'MM_01', 'MM_02', 'MM_03', 'MM_04', 'MM_05', 'CG_02', 'CG_03', 'CG_04', 'CG_05', 'CG_06', 'CG_07', 'CG_08', 'CG_09', 'CG_10', 'CG_11', 'CG_12', 'JI_01', 'JI_02', 'JI_03', 'JI_04', 'JI_05', 'JI_06', 'JI_07', 'JI_08', 'JI_09', 'JI_10', 'JI_11', 'JI_12', 'SS_01', 'SS_02', 'SS_03', 'SS_04', 'SS_05', 'KM_08', 'KM_09', 'KM_10', 'KM_11', 'KM_12', 'ES_06', 'ES_07', 'ES_08', 'ES_09', 'ES_10', 'ES_11', 'ES_12', 'BE_04', 'HS_11', 'HS_12', 'OR_01', 'OR_02', 'LCD_02', 'LCD_03', 'LCD_04', 'LCD_05', 'LCD_06', 'LCD_07', 'LCD_08', 'LCD_09', 'LCD_10', 'LCD_11', 'LCD_12', 'BN_01', 'BN_02', 'BN_03', 'BN_04', 'BN_05', 'BN_06', 'BN_07', 'BN_08', 'BN_09', 'BN_10', 'BN_11', 'BN_12']
Popularity — mean=65.1  sd=17.2  min=10  max=92


---
## Step 9 — Combine Results
Merges all model similarity scores, Spotify data, and segment results  
into a single flat CSV ready for the R analysis.  
**Output:** `results/master_results.csv`

In [11]:
combine_results = load_script("combine_results", "09_combine_results.py")

master_df = combine_results.combine()

print(f"\nmaster_results.csv — {master_df.shape[0]} rows × {master_df.shape[1]} columns")
master_df.head(10)

16:21:39 | INFO | combine_results | LMC data: MuLan=436, CLAP=436, MERT+SBERT=436
16:21:39 | INFO | combine_results | Spotify: 363 found
16:21:39 | INFO | combine_results | Segment data: 383 songs
16:21:39 | INFO | combine_results | 
════════════════════════════════════════════════════════════
16:21:39 | INFO | combine_results | Master dataset: 441 songs across 34 artists
16:21:39 | INFO | combine_results |   With popularity:      363/441
16:21:39 | INFO | combine_results |   With MuLan LMC:       436/441
16:21:39 | INFO | combine_results |   With CLAP LMC:        436/441
16:21:39 | INFO | combine_results |   With MERT+SBERT LMC:  436/441
16:21:39 | INFO | combine_results |   With segment LMC:     383/441
16:21:39 | INFO | combine_results | 
LMC descriptives (MuLan):
16:21:39 | INFO | combine_results | count    436.000000
mean       0.331827
std        0.093126
min        0.011440
25%        0.267559
50%        0.333394
75%        0.394678
max        0.643509
16:21:39 | INFO | combine_


master_results.csv — 441 rows × 43 columns


,song_id,title,artist_name,artist_code,genre,orientation,popularity,lmc_mulan,lmc_clap,lmc_mert_sbert,...,lmc_clap_z,lmc_mert_sbert_z,narrative,genre_country,genre_electronic,genre_folk,genre_folk-rock,genre_hip-hop,genre_pop,genre_psychedelic-electronic
0,RTJ_01,Run the Jewels,Run the Jewels,RTJ,hip-hop,narrative,52.0,0.347461,0.209607,-0.114023,...,1.352596,-2.867184,1,False,False,False,False,True,False,False
1,RTJ_02,Banana Clipper,Run the Jewels,RTJ,hip-hop,narrative,42.0,0.335946,0.248445,-0.046507,...,1.741231,-0.631636,1,False,False,False,False,True,False,False
2,RTJ_03,36 Inch Chain,Run the Jewels,RTJ,hip-hop,narrative,10.0,0.382107,0.426227,-0.019562,...,3.520206,0.260584,1,False,False,False,False,True,False,False
3,RTJ_04,DDHF,Run the Jewels,RTJ,hip-hop,narrative,38.0,0.337209,0.146885,-0.040884,...,0.724974,-0.445425,1,False,False,False,False,True,False,False
4,RTJ_05,Sea Legs,Run the Jewels,RTJ,hip-hop,narrative,38.0,0.377617,0.175617,-0.090214,...,1.012478,-2.078830,1,False,False,False,False,True,False,False
5,RTJ_06,Job Well Done,Run the Jewels,RTJ,hip-hop,narrative,32.0,0.338646,0.136837,-0.044451,...,0.624422,-0.563554,1,False,False,False,False,True,False,False
6,RTJ_07,No Come Down,Run the Jewels,RTJ,hip-hop,narrative,32.0,0.492247,0.235483,-0.029966,...,1.611525,-0.083923,1,False,False,False,False,True,False,False
7,RTJ_08,Get It,Run the Jewels,RTJ,hip-hop,narrative,40.0,0.416955,0.146924,-0.006813,...,0.725357,0.682715,1,False,False,False,False,True,False,False
8,RTJ_09,Twin Hype Back,Run the Jewels,RTJ,hip-hop,narrative,29.0,0.214881,0.179880,-0.052160,...,1.055137,-0.818791,1,False,False,False,False,True,False,False
9,RTJ_10,A Christmas Fucking Miracle,Run the Jewels,RTJ,hip-hop,narrative,36.0,0.394566,0.283586,-0.062443,...,2.092873,-1.159304,1,False,False,False,False,True,False,False


---
## Step 10 - Lyrics over Time

In [12]:
# ── Step 10: Lyric Timeline ───────────────────────────────────────────────────
lyric_timeline = load_script("lyric_timeline", "10_lyric_timeline.py")

timeline_csv = lyric_timeline.run_lyric_timeline(
    device        = device,
    force         = False,
    artist_filter = None,   # e.g. ["KL", "GD"] to test on a subset first
)

import pandas as pd
tl = pd.read_csv(timeline_csv)
print(f"Timeline: {len(tl):,} lines across {tl['song_id'].nunique()} songs")
print(f"Mean LMC: {tl['lmc'].mean():.4f}  |  SD: {tl['lmc'].std():.4f}")
tl.head(10)

09:44:46 | INFO | lyric_timeline | Processing 436 songs...
09:44:46 | INFO | lyric_timeline | Loading MuQ-MuLan...
09:44:46 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
09:44:46 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/OpenMuQ/MuQ-MuLan-large/2e01c796b71dca71b45251384c04cd7b237c9020/config.json "HTTP/1.1 200 OK"
09:44:46 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-large-msd-iter/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
09:44:46 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/OpenMuQ/MuQ-large-msd-iter/0562a57814f6f8bbd9fdea0a25921a2fce1a841a/config.json "HTTP/1.1 200 OK"
/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  Wei

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
09:44:50 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
09:44:50 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/OpenMuQ/MuQ-MuLan-large/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
09:45:52 | INFO | lyric_timeline | Loading WhisperX on cpu...
09:45:52 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-base/revision/main "HTTP/1.1 200 OK"


2026-04-21 09:45:53 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-04-21 09:45:53 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../opt/anaconda3/lib/python3.13/site-packages/whisperx/assets/pytorch_model.bin`
09:45:54 | INFO | lightning.pytorch.utilities.migration.utils | Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../opt/anaconda3/lib/python3.13/site-packages/whisperx/assets/pytorch_model.bin`
09:45:54 | INFO | lyric_timeline | 
[1/436] RTJ_01 — Run the Jewels
python(57372) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


...

13:25:24 | INFO | lyric_timeline |   [GZ_11] 76 lines, 133 WhisperX words
13:25:26 | INFO | lyric_timeline |   [GZ_11] 1 lines matched and embedded
13:25:26 | INFO | lyric_timeline |   Checkpoint saved (435/436 songs done)
13:25:26 | INFO | lyric_timeline | 
[436/436] GZ_12 — Gorillaz
python(76679) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


2026-04-21 13:25:35 - whisperx.asr - INFO - Detected language: en (0.33) in first 30s of audio


13:25:41 | INFO | lyric_timeline |   [GZ_12] 72 lines, 141 WhisperX words
13:25:43 | INFO | lyric_timeline |   [GZ_12] 1 lines matched and embedded
13:25:43 | INFO | lyric_timeline |   Checkpoint saved (436/436 songs done)
13:25:43 | INFO | lyric_timeline | 
13:25:43 | INFO | lyric_timeline | Timeline complete.
13:25:43 | INFO | lyric_timeline |   Rows    : 2,087
13:25:43 | INFO | lyric_timeline |   Songs   : 366
13:25:43 | INFO | lyric_timeline |   LMC mean: 0.2513
13:25:43 | INFO | lyric_timeline |   LMC sd  : 0.1095
13:25:43 | INFO | lyric_timeline |   Output  : /Users/budge.13/Desktop/Music Analysis/results/lyric_timeline/lyric_timeline.csv


Timeline: 2,087 lines across 366 songs
Mean LMC: 0.2513  |  SD: 0.1095


,song_id,artist_name,genre,orientation,line_idx,line_text,line_words,match_start_s,match_end_s,window_start_s,window_end_s,position_pct,total_duration_s,lmc,match_confidence
0,RTJ_01,Run the Jewels,hip-hop,narrative,0,"Run, run, run, run, run",5,50.413,79.041,45.413,84.041,31.101,208.120,0.233961,0.400
1,RTJ_01,Run the Jewels,hip-hop,narrative,8,Better hide all the snacks and the dough,8,147.829,149.330,142.829,154.330,71.391,208.120,0.283325,0.250
2,RTJ_02,Run the Jewels,hip-hop,narrative,1,"I presented the evidence, eloquent as the Pres...",8,57.016,58.978,52.016,63.978,33.808,171.549,0.161249,0.250
3,RTJ_03,Run the Jewels,hip-hop,narrative,4,Move like Jesus die like a martyr,7,82.183,83.404,77.183,88.404,47.880,172.919,0.160052,0.286
4,RTJ_03,Run the Jewels,hip-hop,narrative,13,"Bad hoes, we in the DR",6,145.648,146.969,140.648,151.969,84.611,172.919,0.201795,0.333
5,RTJ_05,Run the Jewels,hip-hop,narrative,0,Really fell out the lane with this shit,8,27.834,29.957,22.834,34.957,13.008,222.144,0.182380,0.375
6,RTJ_05,Run the Jewels,hip-hop,narrative,1,"Man, print this shit: I'm a misfit",7,31.198,32.659,26.198,37.659,14.373,222.144,0.131243,0.286
7,RTJ_05,Run the Jewels,hip-hop,narrative,3,Born to the next-gen system,5,36.924,37.825,31.924,42.825,16.824,222.144,0.060803,0.400
8,RTJ_05,Run the Jewels,hip-hop,narrative,21,"I don't only bite, but I'm rabid",7,156.167,157.348,151.167,162.348,70.566,222.144,0.336220,0.286
9,RTJ_06,Run the Jewels,hip-hop,narrative,0,"Killer Mike and El-P, fuck boys know the combi...",11,0.411,4.596,0.000,9.596,1.391,180.001,0.234441,0.636


---
## Step 11 - Mood

In [14]:
pip install essentia

python(82237) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 15.8 MB/s eta 0:00:00 MB/s eta 0:00:0101
Note: you may need to restart the kernel to use updated packages.


In [22]:
mood_extractor = load_script("extract_mood", "11_extract_mood.py")
mood_csv = mood_extractor.extract_all_mood(force=False)

16:27:04 | INFO | extract_mood | Resuming — 0 songs already extracted.
16:27:04 | INFO | extract_mood | Extracting features for 440 songs (librosa, no TF)...
16:27:04 | INFO | extract_mood | [1/440] RTJ_01
16:27:14 | INFO | extract_mood | [2/440] RTJ_02
16:27:18 | INFO | extract_mood | [3/440] RTJ_03
16:27:22 | INFO | extract_mood | [4/440] RTJ_04
16:27:26 | INFO | extract_mood | [5/440] RTJ_05
16:27:30 | INFO | extract_mood | [6/440] RTJ_06
16:27:34 | INFO | extract_mood | [7/440] RTJ_07
16:27:38 | INFO | extract_mood | [8/440] RTJ_08
16:27:42 | INFO | extract_mood | [9/440] RTJ_09
16:27:46 | INFO | extract_mood | [10/440] RTJ_10
16:27:50 | INFO | extract_mood |   Checkpoint (10/440 done)
16:27:50 | INFO | extract_mood | [11/440] RTJ_11
16:27:54 | INFO | extract_mood | [12/440] RTJ_12
16:27:58 | INFO | extract_mood | [13/440] RTJ_13
16:28:02 | INFO | extract_mood | [14/440] RTJ_14
16:28:06 | INFO | extract_mood | [15/440] RTJ_15
16:28:10 | INFO | extract_mood | [16/440] KL_01
16:28:14

---
## Quick Sanity Check
A fast inline look at the data before handing off to R:  
LMC distributions per model, popularity by genre cluster, and a raw correlation table.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

df = master_df.copy()
lmc_cols = ["lmc_mulan", "lmc_clap", "lmc_mert_sbert"]
lmc_labels = ["MuQ-MuLan", "LAION-CLAP-Music", "MERT+SBERT"]

genre_order = ["Hip-Hop", "Folk/Country", "Pop", "Electronic"]
genre_map   = {
    "hip-hop": "Hip-Hop",
    "folk-rock": "Folk/Country", "folk": "Folk/Country", "country": "Folk/Country",
    "pop": "Pop",
    "electronic": "Electronic", "psychedelic-electronic": "Electronic",
}
df["genre_cluster"] = df["genre"].map(genre_map).fillna("Other")

COLORS = {"Hip-Hop": "#E63946", "Folk/Country": "#2A9D8F",
          "Pop": "#457B9D", "Electronic": "#9B2335"}

fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Row 1: LMC distributions by model ────────────────────────────────────────
for col_idx, (col, label) in enumerate(zip(lmc_cols, lmc_labels)):
    ax = fig.add_subplot(gs[0, col_idx])
    data = df.dropna(subset=[col])
    for gc, grp in data.groupby("genre_cluster"):
        if gc in COLORS:
            ax.hist(grp[col], bins=12, alpha=0.55, label=gc,
                    color=COLORS.get(gc), edgecolor="white")
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("LMC Score")
    ax.set_ylabel("Count" if col_idx == 0 else "")
    if col_idx == 2:
        ax.legend(fontsize=7, loc="upper left")

# ── Row 2, col 1: Popularity by genre cluster ─────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
pop_data = df.dropna(subset=["popularity", "genre_cluster"])
genre_pops = [pop_data[pop_data["genre_cluster"]==g]["popularity"].dropna().values
              for g in genre_order]
bp = ax2.boxplot(genre_pops, labels=genre_order, patch_artist=True, notch=False)
for patch, gc in zip(bp["boxes"], genre_order):
    patch.set_facecolor(COLORS.get(gc, "grey"))
    patch.set_alpha(0.7)
ax2.set_title("Popularity by Genre Cluster", fontweight="bold")
ax2.set_ylabel("Spotify Popularity")
ax2.tick_params(axis="x", labelsize=8)

# ── Row 2, col 2: LMC (MuLan) vs. Popularity scatter ─────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
scatter_df = df.dropna(subset=["lmc_mulan", "popularity"])
for gc in genre_order:
    grp = scatter_df[scatter_df["genre_cluster"] == gc]
    ax3.scatter(grp["lmc_mulan"], grp["popularity"],
                color=COLORS.get(gc), alpha=0.7, s=40, label=gc)
if len(scatter_df) >= 3:
    z = np.polyfit(scatter_df["lmc_mulan"], scatter_df["popularity"], 1)
    xr = np.linspace(scatter_df["lmc_mulan"].min(), scatter_df["lmc_mulan"].max(), 100)
    ax3.plot(xr, np.polyval(z, xr), color="black", linewidth=1.5, linestyle="--")
    r = np.corrcoef(scatter_df["lmc_mulan"], scatter_df["popularity"])[0,1]
    ax3.set_title(f"LMC (MuLan) vs. Popularity  (r = {r:.3f})", fontweight="bold")
ax3.set_xlabel("LMC Score (MuQ-MuLan)")
ax3.set_ylabel("Spotify Popularity")

# ── Row 2, col 3: Correlation table (text) ────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
ax4.axis("off")
corr_cols = ["popularity"] + lmc_cols
corr_df   = df[corr_cols].dropna()
if len(corr_df) >= 3:
    corr_mat = corr_df.corr().round(3)
    short_names = ["Popularity", "MuLan", "CLAP", "MERT+SBERT"]
    rows = [[short_names[i]] + [f"{corr_mat.iloc[i,j]:.3f}" for j in range(len(corr_cols))]
            for i in range(len(corr_cols))]
    tbl = ax4.table(
        cellText=rows,
        colLabels=["Variable"] + short_names,
        loc="center", cellLoc="center"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.6)
    ax4.set_title("Pearson Correlations", fontweight="bold", pad=12)

fig.suptitle("Musical Congruence — Pipeline Sanity Check",
             fontsize=14, fontweight="bold", y=1.01)
plt.savefig("results/sanity_check.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/sanity_check.png")

---
## Pipeline Summary

In [ ]:
import os
from pathlib import Path
from config import RESULTS_DIR, EMBEDDINGS_DIR

def file_exists(path):
    return "✓" if Path(path).exists() else "✗"

print("Pipeline output summary")
print("─" * 52)
print(f"  {file_exists('lyrics/_manifest.json')}  Lyrics manifest")
print(f"  {file_exists('results/embeddings/mulan/similarities.json')}  MuLan similarities")
print(f"  {file_exists('results/embeddings/clap/similarities.json')}  CLAP similarities")
print(f"  {file_exists('results/embeddings/mert_sbert/similarities.json')}  MERT+SBERT similarities")
print(f"  {file_exists('results/segment_analysis/segment_summary.csv')}  Segment summary")
print(f"  {file_exists('results/spotify_data.json')}  Spotify data")
print(f"  {file_exists('results/master_results.csv')}  master_results.csv  ← R input")
print()

master_path = Path("results/master_results.csv")
if master_path.exists():
    import pandas as pd
    m = pd.read_csv(master_path)
    print(f"  master_results.csv: {len(m)} songs, {m.columns.tolist().count} columns")
    print(f"  Complete (popularity + MuLan): "
          f"{m.dropna(subset=['popularity','lmc_mulan']).shape[0]} songs")
    print()
    print("Next step: open analysis/analysis.R in RStudio and run top-to-bottom.")
else:
    print("  master_results.csv not yet generated — run Steps 1–9 above.")